In [33]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import MultiLabelBinarizer
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

In [34]:
# ==========================================
# 1. 설정 (Configuration)
# ==========================================
# [수정] 인코딩 코드로 만든 .pt 파일 경로
TEST_DATA_PATH = "C:/Users/user/Desktop/IDS_masters/small_CAN_MIRGU/validation_dataset.npz"

# [수정] 학습된 모델 가중치 파일 경로
MODEL_PATH = "C:/Users/user/Desktop/IDS_masters/재호/model4.pth"

BATCH_SIZE = 1024
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 라벨 이름 (결과 리포트용)
LABEL_NAME = {0: "Normal", 1: "DoS", 2: "Fuzzing", 3: "Replay", 4: "Spoofing"}
TARGET_NAMES = [LABEL_NAME[i] for i in range(5)]

In [35]:
# ==========================================
# 2. 데이터셋 클래스 정의 (NpDataset)
# ==========================================
class NpDataset(Dataset):
    def __init__(self, npz_path):
        if not os.path.exists(npz_path) : 
            raise FileNotFoundError(f"❌ 데이터 파일이 없습니다: {npz_path}")
            
        print(f"Loading npz dataset from {npz_path}...")
        data = np.load(npz_path)
        
        self.X = torch.nan_to_num(torch.from_numpy(data['X']).float(), nan=0.0)
        self.y = torch.from_numpy(data['y']).long()
        
        print(f"✅ Data Loaded: X shape={self.X.shape}, y shape={self.y.shape}")

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [36]:
# ==========================================
# 3. 모델 구조 정의 (이거 전체를 복사하세요!)
# ==========================================

# 1. SeqIDS가 사용하는 부품 (이게 없으면 에러남!)
class CausalConv1d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation=1):
        super(CausalConv1d, self).__init__()
        # 미래의 정보를 보지 않도록 패딩 설정 (Causal Padding)
        self.padding = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(in_channels, out_channels, kernel_size, 
                              padding=self.padding, dilation=dilation)

    def forward(self, x):
        x = self.conv(x)
        # 패딩만큼 뒤를 잘라내서 입력과 길이를 맞춤
        if self.padding != 0:
            x = x[:, :, :-self.padding]
        return x

# 2. 사용자님이 보내주신 모델 (메인)
class SeqIDS(nn.Module):
    def __init__(self, num_classes=5, dropout_rate=0.5):
        super(SeqIDS, self).__init__()
        # 위에서 정의한 CausalConv1d를 사용
        self.layer1 = CausalConv1d(9, 32, kernel_size=5)
        self.bn1 = nn.BatchNorm1d(32)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(p=dropout_rate)


        self.layer2 = CausalConv1d(32, 64, kernel_size=3)
        self.bn2 = nn.BatchNorm1d(64)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(p=dropout_rate)
        
        # 각 패킷(time step)별로 클래스를 분류하는 1x1 Conv
        self.classifier = nn.Conv1d(64, num_classes, kernel_size=1)

    def forward(self, x):
        # x: (B, 9, 64)
        x = self.dropout1(self.relu1(self.bn1(self.layer1(x))))
        x = self.dropout2(self.relu2(self.bn2(self.layer2(x))))
        
        logits = self.classifier(x)  # (B, 5, 64)
        return logits

In [37]:
def main():
    print(f"🚀 Device: {DEVICE}")

    # 1. 데이터셋 & 데이터로더 준비 (기존과 동일)
    test_ds = NpDataset(TEST_DATA_PATH)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

    # 2. 모델 준비 및 가중치 로드 (기존과 동일)
    model = SeqIDS(num_classes=5).to(DEVICE)    
    if os.path.exists(MODEL_PATH):
        model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
        print(f"✅ 모델 가중치 로드 완료: {MODEL_PATH}")
    else:
        print(f"❌ 모델 파일을 찾을 수 없습니다: {MODEL_PATH}")
        return

    # 3. 평가 시작
    model.eval()
    all_preds_flat = []
    all_targets_flat = []
    mlb = MultiLabelBinarizer(classes=[0, 1, 2, 3, 4])
    
    # [핵심 수정 1] 전체 통합 혼동 행렬을 0으로 초기화합니다.
    conf_mat = torch.zeros(5, 5)

    print("🚀 패킷 단위(Packet-Level) 추론 시작...")
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(DEVICE)
            logits = model(inputs) 
            preds = logits.argmax(dim=1) # (Batch, 64)

            # [핵심 수정 2] 'y'가 아닌 'labels'를 사용합니다.
            y_cpu = labels.cpu()
            preds_cpu = preds.cpu()

            # [참고] 시현님 코드 부분: 150만 번의 Python 루프는 매우 느리므로 
            # Vectorized 연산을 사용하거나 루프 안에서 labels를 사용합니다.
            for t, p in zip(y_cpu.view(-1), preds_cpu.view(-1)):
                conf_mat[t.long(), p.long()] += 1
            
            # 리스트에 추가 (Multi-label 지표용)
            all_preds_flat.extend(preds_cpu.numpy().flatten().reshape(-1, 1).tolist())
            all_targets_flat.extend(y_cpu.numpy().flatten().reshape(-1, 1).tolist())

    # 4. 최종 지표 계산 (기존과 동일)
    y_true = mlb.fit_transform(all_targets_flat)
    y_pred = mlb.transform(all_preds_flat)

    print("\n" + "="*60)
    print("📊 [Packet-Level] 최종 상세 평가 리포트")
    print("="*60)
    print(classification_report(y_true, y_pred, target_names=TARGET_NAMES, zero_division=0))
    
    # 5. [핵심 수정 3] 통합 혼동 행렬 출력
    print("\n=== 전체 통합 Confusion Matrix ===")
    print(conf_mat.int())
    row_sum = conf_mat.sum(dim=1)
    tp = torch.diag(conf_mat)

    print("\n=== Per-class (Attack) Performance ===")
    for i in [0,1,2,3,4]:
        total_i = int(row_sum[i].item())
        correct_i = int(tp[i].item())
        acc_i = 100.0 * correct_i / total_i if total_i > 0 else 0.0
        print(f"{LABEL_NAME[i]:>10s} : {acc_i:6.2f}%  (correct {correct_i}/{total_i})")
    

if __name__ == "__main__":
    main()

🚀 Device: cuda
Loading npz dataset from C:/Users/user/Desktop/IDS_masters/small_CAN_MIRGU/validation_dataset.npz...
✅ Data Loaded: X shape=torch.Size([548692, 9, 64]), y shape=torch.Size([548692, 64])
✅ 모델 가중치 로드 완료: C:/Users/user/Desktop/IDS_masters/재호/model4.pth
🚀 패킷 단위(Packet-Level) 추론 시작...

📊 [Packet-Level] 최종 상세 평가 리포트
              precision    recall  f1-score   support

      Normal       0.97      0.98      0.97  30453254
         DoS       0.39      0.83      0.53   1175042
     Fuzzing       0.69      1.00      0.81    983694
      Replay       0.00      0.00      0.00         0
    Spoofing       1.00      0.11      0.20   2504298

   micro avg       0.91      0.91      0.91  35116288
   macro avg       0.61      0.58      0.50  35116288
weighted avg       0.95      0.91      0.90  35116288
 samples avg       0.91      0.91      0.91  35116288


=== 전체 통합 Confusion Matrix ===
tensor([[16777216,   137112,   446712,   116989,       41],
        [  181479,   980196,      646,